# Main Paper Figures

Paper-ready figures are ordered according to the manuscript. Figure 1 is the separately prepared conceptual schematic. Detailed lambda responses, the complete four-objective liability results, and distribution-distance controls are generated in the SI notebook.

In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, PercentFormatter

ROOT = Path.cwd()
while not (ROOT / "results").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))

import paper_plot_data_loaders as plot_loaders
importlib.reload(plot_loaders)
from paper_plot_data_loaders import (
    load_reinvent_liability,
    load_guacamol_liability,
    load_guacamol_qed,
    PAPER_SEEDS_10,
)

RESULTS = ROOT / "results"
FIG_DIR = ROOT / "notebooks" / "figures" / "main"
DATA_DIR = FIG_DIR / "plotting_data"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.labelsize": 9.5,
    "axes.titlesize": 10.5,
    "legend.fontsize": 8.5,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.edgecolor": "#111111",
    "axes.linewidth": 0.9,
    "xtick.color": "#111111",
    "ytick.color": "#111111",
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

METHOD_COLORS = {
    "base": "#4B5563",
    "random_neon": "#A3A3A3",
    "positive": "#EE9B00",
    "neon": "#1A759F",
    "cne": "#52B69A",
}
NE_LAMBDA_COLORS = {0.10: "#184E77", 0.25: "#1E6091", 0.50: "#1A759F", 0.75: "#168AAD", 1.00: "#34A0A4"}

def style_ax(ax):
    ax.grid(False)
    ax.tick_params(axis="both", which="major", bottom=True, left=True, length=3.5, width=0.8, color="#111111", labelcolor="#111111")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#111111")
        spine.set_linewidth(0.9)

def save_figure(fig, name: str):
    for suffix in ("png", "svg", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{suffix}", bbox_inches="tight")
    print(FIG_DIR / f"{name}.png")

def label_subplots(fig, labels=None, x=-0.12, y=1.045, axes=None):
    if axes is None:
        axes = [ax for ax in fig.axes if ax.get_visible()]
    if labels is None:
        labels = [chr(ord("a") + i) for i in range(len(axes))]
    for ax, label in zip(axes, labels):
        ax.text(
            x,
            y,
            label,
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontweight="bold",
            fontsize=11.0,
            color="#111111",
            clip_on=False,
        )

def paired_reference_boxplot(
    ax, frame, *, model_col, value_col, reference_col, model_order,
    model_labels, model_colors, reference_order, reference_labels,
):
    positions = np.arange(len(model_order), dtype=float)
    for reference, offset, hatch in zip(reference_order, (-0.17, 0.17), (None, "///")):
        for model_index, model in enumerate(model_order):
            values = frame.loc[
                frame[model_col].eq(model) & frame[reference_col].eq(reference), value_col
            ].dropna().to_numpy(dtype=float)
            if not len(values):
                continue
            artists = ax.boxplot(
                [values], positions=[positions[model_index] + offset], widths=0.28,
                patch_artist=True, showfliers=False,
                medianprops={"color": "#111111", "linewidth": 1.0},
                boxprops={"edgecolor": "#111111", "linewidth": 0.8},
                whiskerprops={"color": "#111111", "linewidth": 0.8},
                capprops={"color": "#111111", "linewidth": 0.8},
            )
            box = artists["boxes"][0]
            box.set_facecolor(model_colors[model])
            box.set_alpha(0.90 if reference == reference_order[0] else 0.62)
            if hatch:
                box.set_hatch(hatch)
    ax.set_xticks(positions, [model_labels[model] for model in model_order], rotation=25, ha="right")
    ax.legend(
        handles=[
            Patch(facecolor="white", edgecolor="#111111", label=reference_labels[reference_order[0]]),
            Patch(facecolor="white", edgecolor="#111111", hatch="///", label=reference_labels[reference_order[1]]),
        ],
        frameon=False, loc="upper left",
    )
    style_ax(ax)



def mean_ci(df: pd.DataFrame, group_cols: list[str], value_col: str) -> pd.DataFrame:
    out = df.groupby(group_cols, observed=True)[value_col].agg(mean="mean", std="std", n="count").reset_index()
    out["sem"] = out["std"] / np.sqrt(out["n"].clip(lower=1))
    out["t_critical"] = out["n"].map(lambda n: float(stats.t.ppf(0.975, n - 1)) if n > 1 else np.nan)
    out["ci95"] = out["t_critical"] * out["sem"]
    return out

## Figure 2: QED Control Experiment

Headline endpoint comparison for the initial global-property experiment. Lambda-response curves are reported in the Supplementary Information.

In [ ]:
qed = load_guacamol_qed()
qed["model"] = qed["model"].astype(str)

QED_SELECTED_MODELS = ["base", "random_neon_lambda_1.0", "positive", "neon_lambda_1.0"]
QED_LABELS = {
    "base": "Base",
    "random_neon_lambda_1.0": "Random NE 1.0",
    "positive": "Positive FT",
    "neon_lambda_1.0": "NE 1.0",
}
QED_COLORS = {
    "base": METHOD_COLORS["base"],
    "random_neon_lambda_1.0": METHOD_COLORS["random_neon"],
    "positive": METHOD_COLORS["positive"],
    "neon_lambda_1.0": NE_LAMBDA_COLORS[1.00],
}
QED_ARCH_ORDER = ["RNN", "Transformer"]
QED_METRICS = [
    ("qed_mean", "Mean QED"),
    ("qed_ge_0.9_fraction", "QED >= 0.9"),
    ("valid_fraction", "Valid-output fraction"),
]

qed_endpoint = qed[qed["model"].isin(QED_SELECTED_MODELS)].copy()
qed_endpoint["method_label"] = qed_endpoint["model"].map(QED_LABELS)
qed_endpoint["method_label"] = pd.Categorical(
    qed_endpoint["method_label"],
    [QED_LABELS[m] for m in QED_SELECTED_MODELS],
    ordered=True,
)

fig, axes = plt.subplots(2, 3, figsize=(11.5, 5.5), constrained_layout=True)
for row, architecture in enumerate(QED_ARCH_ORDER):
    arch_endpoint = qed_endpoint[qed_endpoint["architecture"].eq(architecture)]
    for col, (metric, ylabel) in enumerate(QED_METRICS):
        ax = axes[row, col]
        sns.boxplot(
            data=arch_endpoint,
            x="method_label",
            y=metric,
            order=[QED_LABELS[m] for m in QED_SELECTED_MODELS],
            palette=[QED_COLORS[m] for m in QED_SELECTED_MODELS],
            width=0.58,
            linewidth=0.5,
            fliersize=0,
            ax=ax,
        )
        ax.set_title(f"{architecture}: {ylabel}", loc="left", fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel(ylabel if col == 0 else "")
        if metric.endswith("fraction"):
            ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
        ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
        style_ax(ax)

label_subplots(fig)
save_figure(fig, "figure_2_qed_control")
plt.show()

qed_endpoint.to_csv(DATA_DIR / "figure_2_qed_endpoint.csv", index=False)

## Figure 3: Transformer Scope Ablation

Architecture-development experiment showing why the final Transformer block and output projection were selected for subsequent Transformer liability-removal experiments.

In [ ]:
SCOPE_OBJECTIVE = "Metal-binding motif"
SCOPE_METRIC = "chelator_hit_fraction"
SCOPE_DIR = RESULTS / "objectives"

SCOPE_LABELS = {
    "full": "Full model",
    "no_embeddings_layernorm": "No embeddings / layer norm",
    "last2_blocks_output": "Last 2 blocks + output",
    "last_block_output": "Last block + output",
    "output": "Output only",
}
SCOPE_ORDER = ["full", "no_embeddings_layernorm", "last2_blocks_output", "last_block_output", "output"]
SCOPE_COLORS = {
    "full": NE_LAMBDA_COLORS[0.75],
    "no_embeddings_layernorm": "#B5E48C",
    "last2_blocks_output": "#99D98C",
    "last_block_output": "#52B69A",
    "output": "#D9ED92",
}


def _read_objective_csv(folder: str) -> pd.DataFrame:
    path = SCOPE_DIR / folder / "objective_metrics.csv"
    frame = pd.read_csv(path).copy()
    frame["source_folder"] = folder
    return frame


def _parse_scoped_ne_model(model: str):
    model = str(model)
    if model.startswith("random_") or not model.startswith("neon_"):
        return None
    text = model.removeprefix("neon_")
    if text.startswith("lambda_"):
        return {"scope": "full", "lambda": float(text.removeprefix("lambda_"))}
    if "_lambda_" not in text:
        return None
    scope, lam_text = text.rsplit("_lambda_", 1)
    if scope in SCOPE_LABELS:
        return {"scope": scope, "lambda": float(lam_text)}
    return None

# Clean development experiment: every scope-lambda combination uses the same three seeds.
SCOPE_DEVELOPMENT_SEEDS = [5, 7, 11]
SCOPE_LAMBDAS = [0.1, 0.25, 0.5, 0.75, 1.0]
scope_response = _read_objective_csv("guacamol_transformer_chelator_scope_development")
parsed = scope_response["model"].map(_parse_scoped_ne_model).apply(
    lambda x: pd.Series(x) if x else pd.Series(dtype=float)
)
scope_response = pd.concat([scope_response, parsed], axis=1)
scope_response = scope_response[
    scope_response["scope"].isin(SCOPE_ORDER)
    & scope_response["seed"].isin(SCOPE_DEVELOPMENT_SEEDS)
    & scope_response["lambda"].isin(SCOPE_LAMBDAS)
].copy()
scope_response["scope_label"] = scope_response["scope"].map(SCOPE_LABELS)

cell_counts = scope_response.groupby(["scope", "lambda"], observed=True)["seed"].nunique()
expected_cells = pd.MultiIndex.from_product([SCOPE_ORDER, SCOPE_LAMBDAS], names=["scope", "lambda"])
cell_counts = cell_counts.reindex(expected_cells, fill_value=0)
if not cell_counts.eq(len(SCOPE_DEVELOPMENT_SEEDS)).all():
    missing = cell_counts[cell_counts.ne(len(SCOPE_DEVELOPMENT_SEEDS))]
    raise ValueError(f"Incomplete clean scope experiment; seed counts by scope/lambda:\n{missing}")

scope_response_summary = (
    scope_response.groupby(["scope", "scope_label", "lambda"], observed=True)[
        [SCOPE_METRIC, "valid_fraction"]
    ]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(2, 2, figsize=(9.6, 7.1), constrained_layout=True)

screen_lambda = 1.0
screen_plot = scope_response[np.isclose(scope_response["lambda"], screen_lambda)].copy()
for col, (metric, ylabel) in enumerate([(SCOPE_METRIC, "Metal-binding-motif hit rate"), ("valid_fraction", "Validity")]):
    ax = axes[0, col]
    for scope_index, scope in enumerate(SCOPE_ORDER):
        values = screen_plot.loc[screen_plot["scope"].eq(scope), metric].dropna().to_numpy(float)
        offsets = np.linspace(-0.045, 0.045, len(values)) if len(values) > 1 else np.zeros(len(values))
        ax.scatter(scope_index + offsets, values, s=28, color=SCOPE_COLORS[scope], edgecolor="#111111", linewidth=0.45, zorder=3)
        if len(values):
            ax.plot([scope_index - 0.13, scope_index + 0.13], [values.mean(), values.mean()], color="#111111", lw=1.1, zorder=4)
    ax.set_title(f"Scope screen at λ={screen_lambda:g}", loc="left", fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_xticks(range(len(SCOPE_ORDER)), [SCOPE_LABELS[s] for s in SCOPE_ORDER], rotation=45, ha="right")
    style_ax(ax)

for ax, metric, ylabel in [
    (axes[1, 0], SCOPE_METRIC, "Metal-binding-motif hit rate"),
    (axes[1, 1], "valid_fraction", "Validity"),
]:
    for scope in ["full", "last_block_output"]:
        raw_scope = scope_response[scope_response["scope"].eq(scope)]
        for _, seed_data in raw_scope.groupby("seed", observed=True):
            seed_data = seed_data.sort_values("lambda")
            ax.plot(seed_data["lambda"], seed_data[metric], marker="o", ms=2.8, lw=0.8, color=SCOPE_COLORS[scope], alpha=0.28, zorder=1)
        part = scope_response_summary[scope_response_summary["scope"].eq(scope)].sort_values("lambda")
        ax.plot(part["lambda"], part[metric], marker="o", lw=2.0, color=SCOPE_COLORS[scope], label=SCOPE_LABELS[scope], zorder=3)
    ax.set_title("Lambda response", loc="left", fontweight="bold")
    ax.set_xlabel("λ")
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_xticks(SCOPE_LAMBDAS)
    style_ax(ax)

axes[1, 1].legend(frameon=False, loc="best")
label_subplots(fig)
save_figure(fig, "figure_3_transformer_scope_ablation")
plt.show()

screen_plot.to_csv(DATA_DIR / "figure_3_transformer_scope_screen.csv", index=False)
scope_response.to_csv(DATA_DIR / "figure_3_transformer_scope_response.csv", index=False)

## Figure 4: Representative Liability Removal Across String-Based Generators

Matched metal-binding-motif comparison across the RNN, scoped Transformer, and pretrained REINVENT prior. All four liability objectives and complete lambda responses are reported in the Supplementary Information.

In [ ]:
REPRESENTATIVE_OBJECTIVE = "Metal-binding motif"
GENERATOR_ORDER = ["RNN", "Transformer", "REINVENT prior"]
COMMON_METHOD_ORDER = ["base", "random_ne", "positive", "ne"]
COMMON_METHOD_LABELS = {
    "base": "Base",
    "random_ne": "Random NE 1.0",
    "positive": "Positive FT",
    "ne": "NE 1.0",
}
COMMON_METHOD_COLORS = {
    "base": METHOD_COLORS["base"],
    "random_ne": METHOD_COLORS["random_neon"],
    "positive": METHOD_COLORS["positive"],
    "ne": NE_LAMBDA_COLORS[1.00],
}


def normalize_guacamol_method(model: str) -> str | None:
    model = str(model)
    if model == "base":
        return "base"
    if model == "positive":
        return "positive"
    if model in {
        "neon_lambda_1", "neon_lambda_1.0",
        "neon_last_block_output_lambda_1", "neon_last_block_output_lambda_1.0",
    }:
        return "ne"
    if model in {
        "random_neon_lambda_1", "random_neon_lambda_1.0",
        "random_neon_last_block_output_lambda_1", "random_neon_last_block_output_lambda_1.0",
    }:
        return "random_ne"
    return None


def normalize_reinvent_method(model: str) -> str | None:
    return {
        "base": "base",
        "random_neon_lambda_1": "random_ne",
        "positive": "positive",
        "neon_lambda_1": "ne",
    }.get(str(model))


guacamol = load_guacamol_liability()
reinvent = load_reinvent_liability()

g_endpoint = guacamol[guacamol["objective_label"].eq(REPRESENTATIVE_OBJECTIVE)].copy()
g_endpoint["method"] = g_endpoint["model"].map(normalize_guacamol_method)
g_endpoint = g_endpoint[g_endpoint["method"].isin(COMMON_METHOD_ORDER)].copy()
g_endpoint["generator"] = g_endpoint["architecture"]

r_endpoint = reinvent[reinvent["objective_label"].eq(REPRESENTATIVE_OBJECTIVE)].copy()
r_endpoint["method"] = r_endpoint["model"].map(normalize_reinvent_method)
r_endpoint = r_endpoint[r_endpoint["method"].isin(COMMON_METHOD_ORDER)].copy()
r_endpoint["generator"] = "REINVENT prior"

cross_endpoint = pd.concat([g_endpoint, r_endpoint], ignore_index=True, sort=False)
cross_endpoint["method_label"] = cross_endpoint["method"].map(COMMON_METHOD_LABELS)
cross_endpoint["method_label"] = pd.Categorical(
    cross_endpoint["method_label"],
    [COMMON_METHOD_LABELS[m] for m in COMMON_METHOD_ORDER],
    ordered=True,
)

STRUCTURE_SOURCES = {
    "RNN": {
        "path": RESULTS / "objectives/guacamol_rnn_chelator_removal/analysis/diversity_selected_10seed/diversity_metrics.csv",
        "normalizer": normalize_guacamol_method,
    },
    "Transformer": {
        "path": RESULTS / "objectives/guacamol_transformer_chelator_lastblock_final/analysis/diversity_selected_10seed/diversity_metrics.csv",
        "normalizer": normalize_guacamol_method,
    },
    "REINVENT prior": {
        "path": RESULTS / "external/reinvent4/chelator_replicates/analysis/diversity/diversity_metrics.csv",
        "normalizer": normalize_reinvent_method,
    },
}
structure_frames = []
for generator, source in STRUCTURE_SOURCES.items():
    frame = pd.read_csv(source["path"])
    frame = frame[frame["seed"].isin(PAPER_SEEDS_10)].copy()
    frame["method"] = frame["model"].map(source["normalizer"])
    frame = frame[frame["method"].isin(COMMON_METHOD_ORDER)].copy()
    frame["generator"] = generator
    frame["method_label"] = frame["method"].map(COMMON_METHOD_LABELS)
    structure_frames.append(frame)
cross_structure = pd.concat(structure_frames, ignore_index=True)
cross_structure["method_label"] = pd.Categorical(
    cross_structure["method_label"],
    [COMMON_METHOD_LABELS[m] for m in COMMON_METHOD_ORDER],
    ordered=True,
)

expected = len(PAPER_SEEDS_10)
endpoint_counts = cross_endpoint.groupby(["generator", "method"], observed=True)["seed"].nunique()
structure_counts = cross_structure.groupby(["generator", "method"], observed=True)["seed"].nunique()
if not endpoint_counts.eq(expected).all() or not structure_counts.eq(expected).all():
    raise ValueError(
        "Incomplete representative-liability data.\n"
        f"Endpoint counts:\n{endpoint_counts}\nStructure counts:\n{structure_counts}"
    )

fig, axes = plt.subplots(3, 3, figsize=(11.4, 8.7), constrained_layout=True)
ROW_SPECS = [
    (cross_endpoint, "target_hit_fraction", "Metal-binding-motif hit fraction"),
    (cross_endpoint, "usable_yield", "Fixed-budget usable yield"),
    (cross_structure, "unique_scaffold_fraction", "Unique scaffold fraction"),
]
for col, generator in enumerate(GENERATOR_ORDER):
    for row, (frame, metric, ylabel) in enumerate(ROW_SPECS):
        ax = axes[row, col]
        panel = frame[frame["generator"].eq(generator)]
        sns.boxplot(
            data=panel,
            x="method_label",
            y=metric,
            order=[COMMON_METHOD_LABELS[m] for m in COMMON_METHOD_ORDER],
            palette=[COMMON_METHOD_COLORS[m] for m in COMMON_METHOD_ORDER],
            width=0.58,
            linewidth=0.5,
            fliersize=0,
            ax=ax,
        )
        ax.set_title(generator if row == 0 else "", loc="left", fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel(ylabel if col == 0 else "")
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
        if row == 2:
            ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
        else:
            ax.set_xticklabels([])
        style_ax(ax)

legend_handles = [
    Patch(
        facecolor=COMMON_METHOD_COLORS[method],
        edgecolor="#111111",
        linewidth=0.6,
        label=COMMON_METHOD_LABELS[method],
    )
    for method in COMMON_METHOD_ORDER
]
fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.025),
)
label_subplots(fig, x=-0.13, y=1.04)
save_figure(fig, "figure_4_cross_architecture_metal_binding")
plt.show()

cross_endpoint.to_csv(DATA_DIR / "figure_4_cross_architecture_metal_binding_endpoints.csv", index=False)
cross_structure.to_csv(DATA_DIR / "figure_4_cross_architecture_metal_binding_diversity.csv", index=False)

## Figure 5: SemlaFlow 3D Liability Removal And Sampling Efficiency

Main comparison of standard NE with corrected negative extrapolation (CNE). CNE subtracts a task vector learned from liability-free molecules to remove shared valid-generation changes from the liability-enriched task vector before extrapolation.

In [ ]:
semlaflow_root = RESULTS / "external" / "semlaflow" / "four_liability_joint_replicates" / "analysis"
semlaflow_paper_dir = semlaflow_root / "paper_analysis"
semlaflow_scaffold_dir = semlaflow_root / "usable_scaffolds"

required_paths = {
    "2D endpoints": semlaflow_root / "replicate_metrics.csv",
    "PoseBusters endpoints": semlaflow_root / "posebusters_replicate_metrics.csv",
    "usable scaffolds": semlaflow_scaffold_dir / "usable_scaffold_metrics.csv",
    "FCD": semlaflow_root / "distribution_distance_fcd" / "distribution_distance_metrics.csv",
}
missing = [f"{name}: {path}" for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing SemlaFlow analysis outputs:\n" + "\n".join(missing))

semla_endpoints = pd.read_csv(required_paths["2D endpoints"])
semla_pose = pd.read_csv(required_paths["PoseBusters endpoints"])
semla_scaffolds = pd.read_csv(required_paths["usable scaffolds"])
semla_fcd = pd.read_csv(required_paths["FCD"])
semla_endpoints = semla_endpoints[semla_endpoints["seed"].isin(PAPER_SEEDS_10)].copy()
semla_pose = semla_pose[semla_pose["seed"].isin(PAPER_SEEDS_10)].copy()
semla_scaffolds = semla_scaffolds[semla_scaffolds["seed"].isin(PAPER_SEEDS_10)].copy()
semla_fcd = semla_fcd[semla_fcd["seed"].isin(PAPER_SEEDS_10)].copy()

SEMLA_POSE_MODEL_MAP = {
    "base": "base",
    "random_tuned": "random_tuned",
    "positive_tuned": "positive_tuned",
    "bad_tuned": "bad_tuned",
    "standard_ne_2p5": "full_model_neon_lambda_2p5",
    "random_ne_2p5": "full_model_random_neon_lambda_2p5",
    "norm_matched_random_corrected_ne_2p5": "full_model_norm_matched_random_corrected_neon_lambda_2p5",
    "positive_corrected_ne_2p5": "full_model_positive_corrected_neon_lambda_2p5",
    "standard_ne_4": "full_model_neon_lambda_4",
    "random_ne_4": "full_model_random_neon_lambda_4",
    "norm_matched_random_corrected_ne_4": "full_model_norm_matched_random_corrected_neon_lambda_4",
}
semla_pose["model"] = semla_pose["model"].map(SEMLA_POSE_MODEL_MAP).fillna(semla_pose["model"])

SEMLA_METHOD_ORDER = [
    "base",
    "full_model_random_neon_lambda_2p5",
    "positive_tuned",
    "full_model_neon_lambda_2p5",
    "full_model_positive_corrected_neon_lambda_2p5",
]
SEMLA_VIABLE_ORDER = [
    "base",
    "positive_tuned",
    "full_model_neon_lambda_2p5",
    "full_model_positive_corrected_neon_lambda_2p5",
]
SEMLA_FCD_ORDER = SEMLA_VIABLE_ORDER.copy()
SEMLA_LABELS = {
    "base": "Base",
    "full_model_random_neon_lambda_2p5": "Random NE",
    "positive_tuned": "Positive FT",
    "full_model_neon_lambda_2p5": "Standard NE",
    "full_model_positive_corrected_neon_lambda_2p5": "CNE",
}
SEMLA_COLORS = {
    "base": METHOD_COLORS["base"],
    "full_model_random_neon_lambda_2p5": METHOD_COLORS["random_neon"],
    "positive_tuned": METHOD_COLORS["positive"],
    "full_model_neon_lambda_2p5": NE_LAMBDA_COLORS[0.50],
    "full_model_positive_corrected_neon_lambda_2p5": "#52B69A",
}

def semla_boxplot(ax, frame, metric, order, ylabel, *, percent=False):
    plot = frame[frame["model"].isin(order)].copy()
    if percent:
        plot[metric] = 100.0 * plot[metric]
    sns.boxplot(
        data=plot,
        x="model",
        hue="model",
        y=metric,
        order=order,
        hue_order=order,
        palette=SEMLA_COLORS,
        dodge=False,
        legend=False,
        width=0.50,
        linewidth=0.8,
        fliersize=0,
        ax=ax,
    )
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.set_xticks(
        np.arange(len(order)),
        labels=[SEMLA_LABELS[model] for model in order],
        rotation=30,
        ha="right",
    )
    style_ax(ax)

fig = plt.figure(figsize=(14.2, 7.2), constrained_layout=True)
grid = fig.add_gridspec(2, 4)
ax_liability = fig.add_subplot(grid[0, 0])
ax_validity = fig.add_subplot(grid[0, 1])
ax_posebusters = fig.add_subplot(grid[0, 2])
ax_usable_3d = fig.add_subplot(grid[0, 3])
ax_fcd = fig.add_subplot(grid[1, 0])
ax_scaffold_diversity = fig.add_subplot(grid[1, 1])
ax_scaffold_yield = fig.add_subplot(grid[1, 2])
ax_efficiency = fig.add_subplot(grid[1, 3])

semla_boxplot(
    ax_liability,
    semla_endpoints,
    "four_liability_hit_fraction",
    SEMLA_METHOD_ORDER,
    "Liability-hit fraction (%)",
    percent=True,
)
ax_liability.set_title("Liability removal", loc="left", fontweight="bold")
ax_liability.yaxis.set_major_locator(MaxNLocator(integer=True))

semla_boxplot(
    ax_validity,
    semla_endpoints,
    "valid_fraction",
    SEMLA_METHOD_ORDER,
    "RDKit-valid graph yield (%)",
    percent=True,
)
ax_validity.set_title("2D graph validity", loc="left", fontweight="bold")
ax_validity.yaxis.set_major_locator(MaxNLocator(nbins=5))

semla_boxplot(
    ax_posebusters,
    semla_pose,
    "posebusters_all_checks_fraction_of_rdkit_valid",
    SEMLA_METHOD_ORDER,
    "PoseBusters pass among RDKit-valid (%)",
    percent=True,
)
ax_posebusters.set_title("Conditional 3D quality", loc="left", fontweight="bold")
ax_posebusters.yaxis.set_major_locator(MaxNLocator(integer=True))

semla_boxplot(
    ax_usable_3d,
    semla_pose,
    "usable_3d_yield",
    SEMLA_METHOD_ORDER,
    "3D usable yield (%)",
    percent=True,
)
ax_usable_3d.set_title("Practical 3D yield", loc="left", fontweight="bold")
ax_usable_3d.yaxis.set_major_locator(MaxNLocator(integer=True))

scaffold_3d = semla_scaffolds[semla_scaffolds["scope"].eq("3d_usable")].copy()
semla_boxplot(
    ax_scaffold_diversity,
    scaffold_3d,
    "unique_scaffold_fraction_among_usable",
    SEMLA_VIABLE_ORDER,
    "Unique scaffolds / 3D-usable molecules (%)",
    percent=True,
)
ax_scaffold_diversity.set_title("Scaffold diversity", loc="left", fontweight="bold")
ax_scaffold_diversity.yaxis.set_major_locator(MaxNLocator(integer=True))

semla_boxplot(
    ax_scaffold_yield,
    scaffold_3d,
    "unique_usable_scaffold_yield",
    SEMLA_VIABLE_ORDER,
    "Unique 3D-usable scaffolds / sampled (%)",
    percent=True,
)
ax_scaffold_yield.set_title("Usable scaffold yield", loc="left", fontweight="bold")
ax_scaffold_yield.yaxis.set_major_locator(MaxNLocator(integer=True))

semla_boxplot(
    ax_efficiency,
    scaffold_3d,
    "samples_to_fixed_scaffold_target",
    SEMLA_VIABLE_ORDER,
    "Samples to 2,500 3D-usable scaffolds",
)
ax_efficiency.set_title("Fixed-target efficiency", loc="left", fontweight="bold")
ax_efficiency.yaxis.set_major_locator(MaxNLocator(integer=True))

fcd_references = ["base", "liability_free_base"]
fcd_labels = {"base": "Raw base", "liability_free_base": "Liability-free base"}
fcd_plot = semla_fcd[
    semla_fcd["model"].isin(SEMLA_FCD_ORDER)
    & semla_fcd["reference"].isin(fcd_references)
].copy()
if fcd_plot["fcd"].isna().any():
    raise ValueError("SemlaFlow FCD table contains missing values")

paired_reference_boxplot(
    ax_fcd, fcd_plot, model_col="model", value_col="fcd", reference_col="reference",
    model_order=SEMLA_FCD_ORDER, model_labels=SEMLA_LABELS, model_colors=SEMLA_COLORS,
    reference_order=fcd_references, reference_labels=fcd_labels,
)
ax_fcd.set_ylabel("FCD to reference")
ax_fcd.set_title("Chemical-space shift", loc="left", fontweight="bold")

legend_handles = [
    Patch(
        facecolor=SEMLA_COLORS[model],
        edgecolor="#111111",
        linewidth=0.6,
        label=SEMLA_LABELS[model],
    )
    for model in SEMLA_METHOD_ORDER
]
fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=6,
    frameon=False,
    bbox_to_anchor=(0.5, 1.025),
)
label_subplots(fig, x=-0.10, y=1.04)
save_figure(fig, "figure_5_semlaflow_3d_liability")
plt.show()

semla_endpoints[semla_endpoints["model"].isin(SEMLA_METHOD_ORDER)].to_csv(
    DATA_DIR / "figure_5_semlaflow_2d_endpoints.csv", index=False
)
semla_pose[semla_pose["model"].isin(SEMLA_METHOD_ORDER)].to_csv(
    DATA_DIR / "figure_5_semlaflow_posebusters_endpoints.csv", index=False
)
scaffold_3d[scaffold_3d["model"].isin(SEMLA_VIABLE_ORDER)].to_csv(
    DATA_DIR / "figure_5_semlaflow_usable_scaffolds.csv", index=False
)
fcd_plot.to_csv(DATA_DIR / "figure_5_semlaflow_fcd.csv", index=False)